# Gordon Ranschburg 2021 Dataset

This notebook processes the Ranschburg spacing experiment data from `GordonRanschburg_Exp1B_Data.csv`.

**Dataset characteristics:**
- Serial recall of 6-letter sequences
- `Lag` field indicates spacing between item repetitions:
  - lag = 0, 1, 2, 3: repetition with that many intervening items
  - lag = 4: no repetition (control trials)
- `Position` field shows which positions contain the repeated item (e.g., "2-5")
- `Condition` field (1, 2, 3) for experimental counterbalancing

**Output:** `RepeatedRecallsGordonRanschburg2021.h5`

In [ ]:
from typing import TypedDict
from jaxcmr.typing import Integer, Array, NotRequired

class RecallDataset(TypedDict):
    """
    A typed dictionary representing a dataset for free or serial recall experiments.
    Each key maps to a 2D integer array of shape (n_trials, ?).
    Rows correspond to trials; columns vary by field.
    Zeros are used to indicate unused or padding entries, with values starting from 1.
    Intrusions are marked with -1.

    Required fields:
        - subject:       Subject IDs (one per trial).
        - listLength:    The length of the list presented in each trial.
        - pres_itemnos:  Within-list item numbers (1-based indices; 0 indicates padding).
        - recalls:       Within-list item numbers for recalled items
                         (1-based indices; 0 indicates padding; -1 indicates intrusion).
    """

    subject: Integer[Array, "n_trials 1"]
    listLength: Integer[Array, "n_trials 1"]
    pres_itemnos: Integer[Array, "n_trials num_presented"]
    recalls: Integer[Array, "n_trials num_recalled"]
    pres_itemids: Integer[Array, "n_trials num_presented"]
    rec_itemids: NotRequired[Integer[Array, "n_trials num_recalled"]]
    session: NotRequired[Integer[Array, "n_trials 1"]]
    lag: NotRequired[Integer[Array, "n_trials 1"]]
    condition: NotRequired[Integer[Array, "n_trials 1"]]

In [ ]:
import pandas as pd
import numpy as np
from jaxcmr.helpers import save_dict_to_hdf5

df = pd.read_csv("data/raw/GordonRanschburg_Exp1B_Data.csv")
print(f"Total rows: {len(df)}")
print(f"Unique subjects: {df['Subject'].nunique()}")
print(f"Lag distribution:\n{df['Lag'].value_counts().sort_index()}")
df.head()

In [ ]:
def letter_id(ch: str) -> int:
    """Return 1..26 for A..Z (case-insensitive); 0 if not a letter."""
    x = ord(ch.upper()) - 64
    return x if 1 <= x <= 26 else 0


# 1) Count trials
n_trials = len(df)

# 2) Determine max lengths for array allocation
max_studied_len = int(df['Word'].str.len().max())
max_recalled_len = int(df['WordTyped'].str.len().max())

print(f"Max studied length: {max_studied_len}")
print(f"Max recalled length: {max_recalled_len}")

# 3) Allocate numpy arrays
subject      = np.zeros((n_trials, 1), dtype=int)
listLength   = np.zeros((n_trials, 1), dtype=int)
pres_itemnos = np.zeros((n_trials, max_studied_len), dtype=int)
recalls      = np.zeros((n_trials, max_recalled_len), dtype=int)
pres_itemids = np.zeros((n_trials, max_studied_len), dtype=int)
rec_itemids  = np.zeros((n_trials, max_recalled_len), dtype=int)
session      = np.zeros((n_trials, 1), dtype=int)
lag          = np.zeros((n_trials, 1), dtype=int)
condition    = np.zeros((n_trials, 1), dtype=int)

# 4) Populate row by row
for i in range(n_trials):
    subject[i, 0] = df.loc[i, 'Subject']
    session[i, 0] = df.loc[i, 'BlockNum']
    lag[i, 0] = df.loc[i, 'Lag']
    condition[i, 0] = df.loc[i, 'Condition']

    # Parse studied sequence
    studied_seq = str(df.loc[i, 'Word'])
    ll = len(studied_seq)
    listLength[i, 0] = ll

    # Fill pres_itemnos - track repeated items
    counter = 0
    for j, ch in enumerate(studied_seq):
        pres_itemids[i, j] = letter_id(ch)
        
        # Check if this letter appeared earlier in the sequence
        previous_position = studied_seq[:j].find(ch)
        if previous_position == -1:
            counter += 1
            pres_itemnos[i, j] = counter
        else:
            # Repeated item - use the first occurrence's position
            pres_itemnos[i, j] = previous_position + 1

    # Parse recall sequence - preserve intrusions as -1
    recall_seq = list(str(df.loc[i, 'WordTyped']))
    
    for recall_idx, ch in enumerate(recall_seq):
        rec_itemids[i, recall_idx] = letter_id(ch)
        
        # Find matching study positions
        possible_positions = [
            pos for pos, let in enumerate(studied_seq) if let == ch
        ]
        
        if possible_positions:
            # Record the first (1-based) position
            recalls[i, recall_idx] = possible_positions[0] + 1
        else:
            # Intrusion - letter not in study list
            recalls[i, recall_idx] = -1

# 5) Pack into dictionary
recall_dataset = {
    "subject":      subject,
    "listLength":   listLength,
    "pres_itemnos": pres_itemnos,
    "recalls":      recalls,
    "pres_itemids": pres_itemids,
    "rec_itemids":  rec_itemids,
    "session":      session,
    "lag":          lag,
    "condition":    condition,
}

print("\nDataset shapes:")
for key in recall_dataset:
    print(f"  {key}: {recall_dataset[key].shape}")

In [ ]:
# Verify repetition structure
print("Sample trials by lag:")
for lag_val in range(5):
    mask = lag.flatten() == lag_val
    if mask.any():
        idx = np.where(mask)[0][0]
        print(f"\nLag {lag_val}:")
        print(f"  pres_itemnos: {pres_itemnos[idx]}")
        print(f"  recalls:      {recalls[idx]}")
        print(f"  Word:         {df.loc[idx, 'Word']}")
        print(f"  Position:     {df.loc[idx, 'Position']}")

In [ ]:
# Save to HDF5
output_path = "data/RepeatedRecallsGordonRanschburg2021.h5"
save_dict_to_hdf5(recall_dataset, output_path)
print(f"Saved to {output_path}")